# RAGAS Framework Interview Questions & Answers

---

## 1. What is RAGAS and what metrics does it provide?

**Answer:** RAGAS (Retrieval Augmented Generation Assessment) is a framework for evaluating RAG pipeline quality. It provides four key metrics:

- **Faithfulness**: Measures if the answer uses only information from the retrieved context
- **Answer Relevancy**: Measures if the answer is relevant to the question asked
- **Context Precision**: Measures how well-ordered and relevant the retrieved chunks are
- **Context Recall**: Measures if the ground truth can be derived from the retrieved context (requires ground truth)

---

## 2. How is the RAGASEvaluationService implemented in this codebase?

**Answer:** The service is in `app/services/ragas_eval_service.py` with:

- `RAGASEvaluationService` class for running evaluations
- `evaluate_single()` - evaluates one question
- `evaluate_dataset()` - evaluates multiple questions
- `_compute_ragas_metrics()` - uses actual RAGAS library
- `_fallback_metrics()` - provides basic metrics if RAGAS fails

---

## 3. What are the API endpoints for RAG evaluation?

**Answer:** Two admin endpoints in `app/routers/admin.py`:

```python
POST /api/v1/admin/evaluate/sample
# Body: {"question": "...", "ground_truth": "...", "top_k": 5}

POST /api/v1/admin/evaluate/dataset  
# Body: {"dataset": [...], "top_k": 5}
```

---

## 4. Why is ground_truth optional for context_recall?

**Answer:** Context recall measures how well the retrieved context covers the expected answer. Without ground truth, there's no reference to compare against, so this metric cannot be calculated.

---

## 5. What fallback metrics are used when RAGAS fails?

**Answer:** When RAGAS computation fails, the service falls back to:

- **Faithfulness**: Checks if context content appears in the answer
- **Answer Relevancy**: Word overlap between question and answer
- **Context Precision**: Overlap between question words and context

---

## 6. How do you run a single evaluation programmatically?

**Answer:**

```python
from app.services.ragas_eval_service import evaluate_rag_sample

result = await evaluate_rag_sample(
    question="What is the maintenance interval?",
    ground_truth="5000 operating hours or 1 year",
    top_k=5
)
# Returns: {question, answer, ground_truth, metrics, error}
```

---

## 7. What data structures hold evaluation results?

**Answer:**

- `EvaluationResult` dataclass - single sample result (question, answer, contexts, metrics)
- `EvaluationSummary` dataclass - aggregate results (mean/std of all metrics, per-sample results)

---

## 8. How is the evaluation dataset structured?

**Answer:**

```python
dataset = [
    {"question": "What is X?", "ground_truth": "Expected answer"},
    {"question": "How to do Y?", "ground_truth": "Expected answer 2"},
]
```

---

## 9. What dependencies are required for RAGAS?

**Answer:** Added to `requirements.txt`:

```python
ragas==0.1.9
datasets==2.18.0
pandas>=2.0.0
```

---

## 10. What is the difference between context_precision and context_recall?

**Answer:**

- **Context Precision**: Are the top-ranked chunks actually relevant? (Order matters)
- **Context Recall**: Did we retrieve enough relevant information to answer the question? (Coverage matters)

---

## 11. How does the evaluation handle errors?

**Answer:** 

- Individual sample errors are captured in `EvaluationResult.error`
- Failed samples are tracked in `EvaluationSummary.failed_samples`
- Aggregate metrics only include successful samples

---

## 12. What LLM and embeddings are used for evaluation?

**Answer:** The service reuses:

- `LLMService` for LLM calls (AzureChatOpenAI via LangChain)
- `EmbedService` for embeddings (Azure OpenAI text-embedding-3-small)

---

## 13. What is the purpose of the top_k parameter?

**Answer:** `top_k` determines how many document chunks are retrieved from the vector store to use as context for answering the question. Default is 5.

---

## 14. How are aggregate metrics calculated?

**Answer:** The service calculates:

- **Mean**: Average of all scores
- **Std**: Standard deviation across scores
- Only valid (non-None) values are included in calculations

---

## 15. What is faithfulness in RAGAS and how is it calculated?

**Answer:** Faithfulness measures whether the answer is grounded only in the retrieved context. It checks if claims in the answer can be derived from the context. In RAGAS, this is calculated using the LLM to verify each statement in the answer against the context.

---

## 16. What is answer_relevancy and how is it measured?

**Answer:** Answer Relevancy measures how well the answer addresses the question. It's calculated by:

1. Generating multiple questions from the answer
2. Computing similarity between original question and generated questions
3. Higher similarity = more relevant answer

---

## 17. How is context_precision different from simple retrieval precision?

**Answer:** Context precision considers the ranking/order of retrieved chunks. Even if relevant chunks are retrieved, if they're ranked low, the precision score decreases. It penalizes when relevant content appears after irrelevant content.

---

## 18. What is the evaluation flow in the RAG pipeline?

**Answer:**

1. Receive question → Clean query
2. Generate embedding → Search vector store
3. Retrieve top_k chunks → Build context
4. Invoke LLM chain → Get answer
5. Compute RAGAS metrics → Return evaluation

---

## 19. Can RAGAS evaluation run without Azure OpenAI?

**Answer:** Yes, the implementation has fallback metrics that don't require LLM calls. These use simple text overlap algorithms. However, full RAGAS metrics need an LLM for evaluating faithfulness and answer relevancy.

---

## 20. How do you interpret RAGAS scores?

**Answer:**

| Score Range | Interpretation |
|-------------|---------------|
| 0.8 - 1.0 | Excellent |
| 0.6 - 0.8 | Good |
| 0.4 - 0.6 | Moderate |
| 0.0 - 0.4 | Poor |

- **Faithfulness < 0.5**: Answer may hallucinate
- **Answer Relevancy < 0.5**: Answer doesn't address question
- **Context Precision < 0.5**: Retrieval ranking needs improvement
- **Context Recall < 0.5**: Not enough relevant context retrieved
